# 00 — From paper replication to the effective BKT baseline

## Purpose

This notebook reconstructs the paper's MathDial Bayesian Knowledge Tracing
pipeline and explains how the project arrived at its effective correctness-only
baseline. It is self-contained: a reader should not need to inspect the scripts
to understand the data, the two implementations, or the final decision.

The argument has three stages:

1. State and audit the paper's data and evaluation contract.
2. Apply it to **Part 1**, the attempted pyBKT replication, and expose the local
   numerical failure and every fallback it requires.
3. Apply the same contract to **Part 2**, a robust NumPy BKT whose design choices
   follow directly from the Part 1 failure.

| Metric | Paper's reported MathDial BKT |
| --- | ---: |
| Accuracy | 0.6071 |
| AUC | 0.6419 |
| Binary F1 (correct = positive) | 0.5671 |

Part 1 is a replication diagnostic. Part 2 is the effective baseline. Part 2 is
not claimed to be the paper's exact estimator; its deliberate differences are
listed and justified below.


## 1. The paper's data and evaluation contract

The original BKT path performs these operations in order:

1. Use MathDial's fixed annotated train and test CSVs.
2. Keep dialogues whose typical-confusion and typical-interaction scores are at
   least 1. This is a no-op on the released annotated CSVs, but is retained.
3. Apply the annotations to dialogue turns:
   - skip failed annotation dialogues;
   - leave a student-initiated turn 0 unlabelled;
   - make turns without KCs `correctness=None` and exclude them;
   - override the final tagged turn using MathDial's `self_correctness`.
4. Retain only dialogues with at least two tagged turns. With fewer than two
   turns there is no next-step target after the first label is excluded.
5. Expand each true turn into one pseudo-observation per KC. Every KC on the same
   turn receives the same correctness label.
6. Fit BKT by KC, treating each `(dialogue, KC)` history as an independent
   sequence and pooling histories for the same KC across dialogues.
7. At test time, average per-KC probabilities back to one true-turn probability.
8. Exclude the first tagged turn of every test dialogue.
9. Report accuracy, ROC AUC, and binary F1 with correct (`1`) as the positive
   class. Hard predictions use `np.round`.

This sequence is the contract used to compare the paper, Part 1, and Part 2.
Any modelling departure is reported explicitly.


In [ ]:
import os
import sys
import json
import time
from copy import deepcopy
from ast import literal_eval
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

_here = Path.cwd()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "data" / "annotated").exists():
        os.chdir(_candidate)
        break
else:
    raise FileNotFoundError(f"Could not find data/annotated above {_here}")

sys.path.insert(0, str(Path.cwd() / "extension"))
from scripts import filtering, bkt, bkt_paper as bp

DATA = Path("data/annotated")
CONVERTERS = {
    name: literal_eval for name in ["dialogue", "annotation", "meta_data"]
}

def read_raw(split):
    return pd.read_csv(
        DATA / f"mathdial_{split}_atc.csv", converters=CONVERTERS
    )

train_raw = read_raw("train")
test_raw = read_raw("test")
print("working directory:", Path.cwd())
print("raw dialogues — train:", len(train_raw), "test:", len(test_raw))


## 2. Construct and audit the shared prepared population

Part 2 needs a long table with one row per `(dialogue, turn, KC)`. It is built
through the paper's annotation function rather than by reading the annotation
dictionary directly. Direct reading would incorrectly turn `None` into false and
would miss the final-turn override.

The helper records each removal. It deep-copies samples because annotation
application mutates the nested dialogue structure; the raw frames remain pristine
for the independent Part 1 run.


In [ ]:
def prepare_long(raw, split):
    typical = bp.apply_typical_filter(raw, typical_cutoff=1)
    usable, n_failed = filtering.drop_failed_annotations(typical)

    rows = []
    n_short = n_kept = n_tagged = 0
    for dialogue_idx, sample in usable.iterrows():
        dialogue = bp.apply_annotations(deepcopy(sample.to_dict()))
        if not dialogue:
            continue
        tagged = [
            turn for turn in dialogue
            if turn["correct"] is not None and turn["kcs"]
        ]
        if len(tagged) < 2:
            n_short += 1
            continue

        n_kept += 1
        n_tagged += len(tagged)
        for turn in tagged:
            label = int(bool(turn["correct"]))
            for kc in turn["kcs"]:
                rows.append((dialogue_idx, turn["turn"], label, str(kc)))

    long_df = pd.DataFrame(
        rows, columns=["dialogue_idx", "turn", "correct", "kc"]
    )
    audit = {
        "split": split,
        "raw_dialogues": len(raw),
        "after_typical": len(typical),
        "failed_removed": n_failed,
        "fewer_than_2_tagged_removed": n_short,
        "dialogues_kept": n_kept,
        "tagged_true_turns": n_tagged,
        "pseudo_observations": len(long_df),
        "distinct_kcs": long_df["kc"].nunique(),
        "paper_scored_turns": n_tagged - n_kept,
    }
    return long_df, audit

train_long, train_audit = prepare_long(train_raw, "train")
test_long, test_audit = prepare_long(test_raw, "test")
audit = pd.DataFrame([train_audit, test_audit]).set_index("split")
print(audit.to_string())

assert not train_long["correct"].isna().any()
assert not test_long["correct"].isna().any()
assert set(train_long["correct"].unique()) <= {0, 1}
assert set(test_long["correct"].unique()) <= {0, 1}


### Annotation, pseudo-turn, and scoring details

The validated prepared population is:

| Split | Raw dialogues | Failed removed | <2 tagged removed | Dialogues kept | Pseudo-observations | Paper-scored turns |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| Train | 2,253 | 18 | 185 | 2,050 | 23,953 | 8,398 |
| Test | 595 | 7 | 73 | 515 | 5,788 | 1,985 |

The final-turn override gives MathDial's dialogue-level outcome priority over the
automatic turn label:

| `self_correctness` | Final tagged-turn label |
| --- | --- |
| `"Yes"` | `True` |
| `"No"` | `False` |
| `"Yes, but I had to reveal the answer"` | `None`, then excluded |

The two-turn filter is applied after NA handling, so “two turns” means two turns
with both a usable correctness label and at least one KC.

A correct true turn with three KCs becomes three training rows labelled 1. At
test time, if their probabilities are \(p_1,p_2,p_3\), the scored probability is

\[
p_{\mathrm{turn}}=(p_1+p_2+p_3)/3.
\]

The first tagged turn is removed, leaving `tagged turns - dialogues` scored
turns. F1 is always binary with correct as the positive class; no alternative F1
definitions are tested or selected.

### What is a fallback pseudo-turn?

A fallback pseudo-turn is a per-KC modelling row for which the model cannot use
a reliable KC-specific fit. It is not an additional dialogue turn and it is not
dropped. The row remains part of its original true turn, but its prediction is
produced by a declared substitute. Other KCs on the same turn can still use
their own fitted parameters.

For example, one true turn might be handled as follows:

| KC pseudo-row | Prediction source |
| --- | --- |
| Fractions | Fractions-specific fitted parameters |
| Division | Division-specific fitted parameters |
| Ratios | Fallback because Ratios is degenerate or unseen |

The three probabilities are still averaged into one prediction for the true
turn. A true turn can therefore contain zero, one, or several fallback
pseudo-turns.

Part 1 and Part 2 use different substitutes:

- **Part 1 substitutes a probability.** If pyBKT returns `NaN`, or the KC is
  degenerate or unseen, the row receives the overall training pseudo-turn
  correctness rate (about 0.497). This ignores both KC identity and response
  history. For example, `[0.70, NaN, 0.40]` becomes
  `[0.70, 0.497, 0.40]` before averaging.
- **Part 2 substitutes BKT parameters.** A degenerate or unseen KC receives the
  pooled prior, learning, guess, and slip vector learned from nondegenerate KCs.
  The row then follows the normal sequential BKT prediction and Bayesian update;
  no correctness probability is inserted directly.

In the validated test run, Part 1 has 1,655 fallback pseudo-turns, whereas Part
2 has 24. These are pseudo-row counts, not counts of distinct true turns.


In [ ]:
# Prove that Part 1's builder and Part 2 retain the same population.
parity = []
for split, expected_long, expected_audit in [
    ("train", train_long, train_audit),
    ("test", test_long, test_audit),
]:
    raw_check = read_raw(split)
    typical = bp.apply_typical_filter(raw_check, typical_cutoff=1)
    pseudo, sequences = bp.build_pseudo_turns(typical)
    row = {
        "split": split,
        "part1_rows": len(pseudo),
        "part2_rows": len(expected_long),
        "part1_dialogues": len(sequences),
        "part2_dialogues": expected_audit["dialogues_kept"],
        "minimum_tagged_turns": min(len(s.turn_labels) for s in sequences),
    }
    parity.append(row)
    assert row["part1_rows"] == row["part2_rows"]
    assert row["part1_dialogues"] == row["part2_dialogues"]
    assert row["minimum_tagged_turns"] >= 2

print(pd.DataFrame(parity).set_index("split").to_string())


# Part 1 — Attempt the paper's pyBKT run

Part 1 uses the paper's model call, `Model(seed=221, num_fits=1)`, and now matches
the paper's preparation, two-turn rule, pseudo-turn construction, aggregation,
first-turn exclusion, and metrics.

The local pyBKT fit nevertheless produces invalid parameters. The wrapper must:

1. exclude all-correct and all-incorrect KCs from fitting;
2. use the training correctness base rate for degenerate and unseen KCs;
3. use the same base rate for any pyBKT prediction that is `NaN`;
4. report the pyBKT NaNs and total fallback coverage.

The paper contains no such safeguard. Part 1 is therefore a diagnostic
replication attempt rather than a usable version of the published estimator.


In [ ]:
PAPER = {"accuracy": 0.6071, "auc": 0.6419, "f1": 0.5671}

part1 = bp.evaluate_bkt(
    train_raw,
    test_raw,
    agg="mean-ar",
    inc_first_label=False,
    seed=221,
    num_fits=1,
    typical_cutoff=1,
    apply_typical=True,
    handle_degenerate=True,
)

print("\nPart 1 — pyBKT with reported safeguards")
print(
    f"n={part1['n']}  Acc={part1['accuracy']:.4f}  "
    f"AUC={part1['auc']:.4f}  F1={part1['f1']:.4f}"
)
print(
    f"pyBKT NaN pseudo-rows={part1['n_pybkt_nan']}  "
    f"total base-rate fallback rows={part1['n_base_rate_fallback']} "
    f"of {test_audit['pseudo_observations']}"
)


## Why Part 1 fails

In the validated run, pyBKT returns 1,631 NaN pseudo-row predictions across
54 fitted skills. After adding degenerate and unseen-skill rows, 1,655 of
5,788 test pseudo-observations (28.6%) use the base rate. The resulting
paper-aligned Part 1 result is Acc=0.5264, AUC=0.5399, F1=0.0000.

pyBKT's warning is raised in its M-step while normalising expected initial-state
soft counts:

```python
model["pi_0"] = init_softcounts / np.sum(init_softcounts)
```

If the denominator is zero or already invalid, this becomes `0/0`; initial
mastery becomes `NaN`, followed by `NaN` predictions for that skill.

Degenerate KCs are underidentified and can trigger collapse, but are not the
whole explanation. Part 1 removes degenerate KCs and one-turn dialogues, yet
pyBKT still returns NaNs for fitted nondegenerate skills. Its output therefore
measures pyBKT plus a large base-rate rescue, not the paper model alone.

This creates the requirements for Part 2: preserve standard BKT semantics, use
scaled inference, keep parameters away from singular boundaries, and make the
policy for unestimable KCs explicit.


# Part 2 — Design the effective baseline

Part 2 implements four-parameter, no-forgetting BKT:

| Parameter | Meaning |
| --- | --- |
| `prior` | mastery before the first response |
| `learns` | unmastered-to-mastered transition after an opportunity |
| `guesses` | correct while unmastered |
| `slips` | incorrect while mastered |

Per-dialogue sequences for each KC are fitted jointly using scaled
forward-backward EM. Five deterministic random restarts reduce sensitivity to a
poor local optimum.

## Decision A: parameter clipping

After every M-step, newly estimated parameters are clipped:

| Parameter | Interval |
| --- | --- |
| prior, learning | `[0.001, 0.999]` |
| guess, slip | `[0.001, 0.49]` |

This prevents exact zero/one probabilities from making contradictory observations
impossible or causing zero denominators. The `0.49` cap also keeps guessing and
slipping less likely than their intended alternatives. Clipping can bias an
unconstrained optimum, so boundary counts are reported later.

## Decision B: degenerate and unseen KCs

An all-correct or all-incorrect KC cannot identify four independent parameters.
Part 2 fits only KCs containing both labels, then calculates one complete
parameter vector as the observation-weighted mean of those fits. Degenerate and
unseen KCs receive that vector.

Observation weighting represents a typical training observation. This is
pragmatic partial pooling, not an estimate of a degenerate KC's own parameters.


In [ ]:
kc_stats = train_long.groupby("kc")["correct"].agg(
    n="count", n_correct="sum"
)
degenerate_mask = (
    (kc_stats["n_correct"] == 0)
    | (kc_stats["n_correct"] == kc_stats["n"])
)
degenerate = kc_stats[degenerate_mask]
nondegenerate = kc_stats[~degenerate_mask]

print("training KCs:", len(kc_stats))
print("fitted individually:", len(nondegenerate))
print("assigned fallback:", len(degenerate))
print(
    "degenerate observations:",
    int(degenerate["n"].sum()),
    f"({100 * degenerate['n'].sum() / kc_stats['n'].sum():.2f}%)",
)

t0 = time.time()
fitted = bkt.fit_bkt(
    train_long, n_restarts=5, max_iter=100, seed=221
)
print(f"fit time: {time.time() - t0:.1f}s")
print("KCs available for prediction:", len(fitted.per_skill))
print("fallback:", {k: round(v, 4) for k, v in fitted.fallback.items()})


## Part 2 prediction and evaluation

For mastery belief \(P(L_t)\), guess \(G\), and slip \(S\):

\[
P(C_t)=P(L_t)(1-S)+(1-P(L_t))G.
\]

Part 2 predicts first, observes the response, updates mastery with Bayes' rule,
then applies learning \(T\):

\[
P(L_{t+1})=P(L_t\mid observation)
 +(1-P(L_t\mid observation))T.
\]

Thus the prediction at time \(t\) uses only earlier responses. The paper-aligned
evaluation then averages KCs, removes the first tagged turn, and uses the paper's
fixed metric definitions.


In [ ]:
def turn_metrics(predictions, exclude_first):
    df = predictions.copy()
    df["_turn_num"] = (
        df["turn"].astype(str).str.extract(r"(\d+)").astype(int)
    )
    turns = (
        df.groupby(["dialogue_idx", "_turn_num"])
        .agg(pred=("pred", "mean"), correct=("correct", "first"))
        .reset_index()
    )
    if exclude_first:
        first = turns.groupby("dialogue_idx")["_turn_num"].idxmin()
        turns = turns.drop(index=first)

    y = turns["correct"].to_numpy(dtype=int)
    p = turns["pred"].to_numpy(dtype=float)
    hard = np.round(p).astype(int)
    return {
        "n": len(y),
        "accuracy": accuracy_score(y, hard),
        "auc": roc_auc_score(y, p),
        "f1": f1_score(
            y, hard, average="binary", pos_label=1, zero_division=0
        ),
    }

predictions = fitted.predict_long(test_long)
part2_all = turn_metrics(predictions, exclude_first=False)
part2_paper = turn_metrics(predictions, exclude_first=True)

for name, metrics in [
    ("Part 2 — all turns (diagnostic)", part2_all),
    ("Part 2 — paper-aligned", part2_paper),
]:
    print(name)
    print(
        f"  n={metrics['n']}  Acc={metrics['accuracy']:.4f}  "
        f"AUC={metrics['auc']:.4f}  F1={metrics['f1']:.4f}"
    )


In [ ]:
train_kcs = set(train_long["kc"])
test_kcs = set(test_long["kc"])
fallback_kcs = set(degenerate.index) | (test_kcs - train_kcs)
part2_fallback_rows = int(test_long["kc"].isin(fallback_kcs).sum())

comparison = pd.DataFrame([
    {"implementation": "Paper (reported)", **PAPER,
     "fallback_pseudo_rows": np.nan},
    {"implementation": "Part 1 (pyBKT + safeguards)",
     "accuracy": part1["accuracy"], "auc": part1["auc"], "f1": part1["f1"],
     "fallback_pseudo_rows": part1["n_base_rate_fallback"]},
    {"implementation": "Part 2 (effective baseline)",
     "accuracy": part2_paper["accuracy"], "auc": part2_paper["auc"],
     "f1": part2_paper["f1"],
     "fallback_pseudo_rows": part2_fallback_rows},
]).set_index("implementation")

print(comparison.round(4).to_string())
print("\nPart 2 minus paper:")
print(pd.Series({
    metric: part2_paper[metric] - PAPER[metric]
    for metric in ["accuracy", "auc", "f1"]
}).round(4).to_string())


## Reading the comparison

| Implementation | Accuracy | AUC | Binary F1 | Fallback pseudo-rows |
| --- | ---: | ---: | ---: | ---: |
| Paper (reported) | 0.6071 | 0.6419 | 0.5671 | — |
| Part 1: pyBKT + safeguards | 0.5264 | 0.5399 | 0.0000 | 1,655 |
| Part 2: effective baseline | 0.6050 | 0.6402 | 0.5556 | 24 |

Part 1 and Part 2 now receive the same prepared population and are scored on the
same true turns. Their difference is not caused by NA labels, the final-turn
override, dialogue length, aggregation, or F1 convention.

Part 1 is rejected because a substantial share of its pseudo-observations must
be replaced after pyBKT returns invalid values. Part 2 is selected because it
preserves standard no-forgetting BKT, uses full scaled inference, matches the
paper's data and scoring contract, retains all test observations through an
explicit fallback, and lands close to the paper on all three metrics.

Metric proximity is supporting evidence, not the sole correctness check.
Preparation parity, the standard Bayesian update, explicit fallback coverage,
and exact model reload are also required.


In [ ]:
MODELS = Path("extension/models")
RESULTS = Path("extension/results")
MODELS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

payload = {
    "model_type": "BKT (correctness-only, nondegenerate fit, pooled fallback)",
    "param_names": list(bkt.PARAM_NAMES),
    "per_skill": fitted.per_skill,
    "fallback": fitted.fallback,
    "metadata": {
        "saved_utc": datetime.now(timezone.utc).isoformat(),
        "fit": {"n_restarts": 5, "max_iter": 100, "seed": 221},
        "n_kcs": len(fitted.per_skill),
        "train_observations": len(train_long),
        "test_observations": len(test_long),
        "n_degenerate_kcs": len(degenerate),
        "fallback_policy": (
            "complete observation-weighted parameter vector from "
            "nondegenerate KCs; used for degenerate/unseen KCs"
        ),
        "data_contract": (
            "paper annotation rules; correctness=None excluded; final-turn "
            "override; minimum two tagged turns"
        ),
    },
}
model_path = MODELS / "bkt_original.json"
with open(model_path, "w") as handle:
    json.dump(payload, handle, indent=2)

with open(model_path) as handle:
    loaded = json.load(handle)
reloaded = bkt.FittedBKT(loaded["per_skill"], loaded["fallback"])
p_reload = reloaded.predict_long(test_long)["pred"].to_numpy()
p_original = predictions["pred"].to_numpy()
max_difference = float(np.max(np.abs(p_reload - p_original)))
assert max_difference < 1e-9

reference = pd.DataFrame([
    {"model": "Original BKT (all KCs), all turns",
     **{k: part2_all[k] for k in ["accuracy", "auc", "f1"]}},
    {"model": "Original BKT (all KCs), paper-aligned",
     **{k: part2_paper[k] for k in ["accuracy", "auc", "f1"]}},
])
reference_path = RESULTS / "original_bkt_reference.csv"
reference.to_csv(reference_path, index=False)

params = pd.DataFrame(fitted.per_skill).T[list(bkt.PARAM_NAMES)]
at_boundary = (
    np.isclose(params["prior"], 0.001)
    | np.isclose(params["prior"], 0.999)
    | np.isclose(params["learns"], 0.001)
    | np.isclose(params["learns"], 0.999)
    | np.isclose(params["guesses"], 0.001)
    | np.isclose(params["guesses"], 0.49)
    | np.isclose(params["slips"], 0.001)
    | np.isclose(params["slips"], 0.49)
)

print("saved:", model_path)
print("saved:", reference_path)
print(f"reload max prediction difference: {max_difference:.2e}")
print("KCs with at least one boundary parameter:", int(at_boundary.sum()))
print("Part 2 fallback test pseudo-rows:", part2_fallback_rows)
print("\nparameter summary:")
print(params.describe().round(3).to_string())


# Decision

**Part 2 is the effective correctness-only BKT baseline.**

The decision follows from a visible chain of evidence:

1. The paper's annotation, dialogue-length, pseudo-turn, aggregation, first-turn,
   and metric rules are stated and audited.
2. Part 1 and Part 2 are proven to receive the same prepared population.
3. Part 1 requires extensive base-rate substitution and cannot serve as the
   baseline.
4. Part 2 implements standard BKT with scaled inference, deterministic restarts,
   bounded parameters, and an explicit partial-pooling fallback.
5. The saved model reloads exactly and its paper-aligned metrics use one fixed
   convention.

## What is deliberately different from the paper

- Part 2 uses custom NumPy EM rather than pyBKT.
- It uses five restarts instead of `num_fits=1`.
- It clips parameters after each M-step.
- It assigns weighted pooled parameters to degenerate and unseen KCs.
- In the validated fit, 81 of 138 KCs have at least one parameter at a clipping
  boundary; this signals weak KC-level identification and must remain visible.

These are retained because they make the baseline runnable and auditable. They
must remain visible when later models are compared against the saved Part 2
reference row.
